# Supplementary Data 1: sample rosters

Supplementary Data 1 (`Sample_rosters.xlsx`) lists, for every acquisition of the four datasets, the analysis rosters it belongs to, its recorded specimen and participant links, and the documented exclusions. This notebook rebuilds the workbook as published with v28d (CapScan on 331 acquisitions, 710 in all) and checks it cell by cell against the published file, then builds the v28f workbook (CapScan on 347 acquisitions from 308 specimens and fifteen participants, 726 in all; five CapScan rosters still on 331) and the v30 workbook (every CapScan roster on 347) and recounts every roster number in Supplementary Methods 1. No statistical test is computed here.

Run from the repository root with `python paper.py build SData1`. PREDICT visit codes, the recorded group code and the antibiotic On/Off pair rows are controlled inputs (`FASTALAKE_CONTROLLED_DIR`); without them the public workbook is built and the antibiotic-pair count is skipped.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

<path> UserWarning: FASTALAKE_FONT_DIR is unset or lacks soehne-buch.otf: using the default typeface. Colours, sizes and layout are unchanged; text pixels will differ from the published panels.
  from style import fl_style as fl


## Table v28d

The published workbook, rebuilt from `source_data/SData1/v28d_*.tsv`. Cohorts and Sheet directory are computed (acquisition, specimen and participant counts from the roster sheets). The original producer's final checks are repeated: each Analyses row's `n_MS_acquisitions` equals the included, distinct acquisitions of its membership sheet (unit: acquisition), and each CapScan row's specimen, participant and participant-region counts equal its membership rows. Claim supported: v28d line 489, "analysis-specific acquisition rosters ... for all 710 acquisitions".

In [2]:
v28d, complete = panels.build_v28d(SD)
order = [s[0] for s in panels.SHEETS]
panels.write_workbook(OUT / "Sample_rosters_v28d.xlsx", order, v28d)
panels.write_analysis_index(OUT / "ANALYSIS_INDEX_v28d.tsv", v28d["Analyses"])
checked = panels.validate_counts(v28d, complete)
entity = panels.validate_capscan_entity_counts(v28d)
print(f"sheets {len(order)}; Analyses rows checked {len(checked)}; CapScan entity counts checked {len(entity)}")
for r in panels.cohorts_sheet(v28d):
    print("\t".join(r[:4]))

sheets 34; Analyses rows checked 30; CapScan entity counts checked 30
cohort	acquisitions	specimens	participants
CAMPI	15	Defined SIHUMIx mixture; no individual specimen IDs in retained metadata	Not applicable
Healthy stool	58	Original specimen IDs unavailable; within-design relationships documented	Ten between-donor labels plus a shared experimental donor of unknown overlap
PREDICT	306	300 explicit tube IDs in August metadata; four acquisitions lack tube links	93 recorded August patients; four acquisitions lack participant links
CapScan	331	292	15


## Table v28f

CapScan moves to the 347-acquisition roster: the 331 published rows plus the sixteen participant-15 intestinal-fluid acquisitions recovered by the identifier join fix (upstream `capscan_participant_roster`). Before applying the rules to 347 acquisitions, the notebook regenerates the four affected analyses on the published 331 roster and requires identical rows (positive control). Rebuilt on 347: CAPSCAN_SEARCH_ALL (all acquisitions), CAPSCAN_QUANT_NONEMPTY and CAPSCAN_REGIONAL_DESCRIPTIVE15 (positive total quantified intensity, from `CAPSCAN_eligibility_audit_347.tsv`, cross-checked with `source_data/Fig5/design_347.tsv`), and CAPSCAN_REGIONAL_PAIRED15 (participants with nonempty acquisitions in all five sampling groups; replaces CAPSCAN_REGIONAL_PAIRED14). The specimen-number census is recomputed from the RAW header audit of all 368 CapScan RAW files (including the sixteen participant-15 files) and recounted against the 347 roster. The functional, native and histidine CapScan rosters stay on 331 and are marked in their limitations (OPEN_ITEMS.md). Claim supported: v28d line 501, "347 acquisitions from 308 specimens and fifteen participants".

In [3]:
controls = panels.reproduce_v28d_capscan(v28d)
print("positive controls on the published 331 roster:", controls)
assert all(all(v for k, v in c.items() if k != "n") for c in controls.values())
v28f = panels.build_v28f_all(v28d, SD, REPO)
panels.write_workbook(OUT / "Sample_rosters_v28f.xlsx", order, v28f)
panels.write_analysis_index(OUT / "ANALYSIS_INDEX_v28f.tsv", v28f["Analyses"])
disc_table, n_audited, n_p15_unchanged = panels.discrepancy_census(panels.read_tsv(SD / "CAPSCAN_raw_header_audit.tsv"))
assert disc_table == panels.read_tsv(SD / "CapScan_specimen_number_discrepancies.tsv")   # published census reproduces
disc_v28d = panels.discrepancies(disc_table, v28d["CAPSCAN acquisitions"])
assert disc_v28d[0] == disc_table           # the published in_analysis_roster column reproduces
print(f"RAW files audited {n_audited}; participant-15 RAW files audited with unchanged specimen number {n_p15_unchanged}")
disc_v28f = panels.discrepancies(disc_table, v28f["CAPSCAN acquisitions"])
panels.write_tsv(OUT / "CapScan_specimen_number_discrepancies_v28f.tsv", disc_v28f[0])
checked_f = panels.validate_counts(v28f, complete)
entity_f = panels.validate_capscan_entity_counts(v28f)
changes = panels.diff_builds(v28d, v28f)
for name, c in changes.items():
    print(name, c)
for r in panels.cohorts_sheet(v28f, panels.V28F_COHORT_DESIGN):
    print("\t".join(r[:4]))

positive controls on the published 331 roster: {'CAPSCAN_SEARCH_ALL': {'membership_equal': True, 'nonmembership_equal': True, 'n': 331}, 'CAPSCAN_QUANT_NONEMPTY': {'membership_equal': True, 'nonmembership_equal': True, 'n': 330}, 'CAPSCAN_REGIONAL_DESCRIPTIVE15': {'membership_equal': True, 'nonmembership_equal': True, 'n': 330}, 'CAPSCAN_REGIONAL_PAIRED14': {'membership_equal': True, 'nonmembership_equal': True, 'n': 326}, 'CAPSCAN regional pairs': {'equal': True}}


RAW files audited 368; participant-15 RAW files audited with unchanged specimen number 16
Cohorts {'rows_v28d': 4, 'rows_v28f': 4, 'rows_removed': 1, 'rows_added': 1}
Analyses {'rows_v28d': 30, 'rows_v28f': 30, 'rows_removed': 9, 'rows_added': 9}
CAPSCAN acquisitions {'rows_v28d': 331, 'rows_v28f': 347, 'rows_removed': 0, 'rows_added': 16}
CAPSCAN membership {'rows_v28d': 7599, 'rows_v28f': 7753, 'rows_removed': 2820, 'rows_added': 2974}
CAPSCAN exclusions {'rows_v28d': 140, 'rows_v28f': 139, 'rows_removed': 1, 'rows_added': 0}
CAPSCAN MS nonmembership {'rows_v28d': 188, 'rows_v28f': 200, 'rows_removed': 5, 'rows_added': 17}
CAPSCAN ID crosswalk {'rows_v28d': 331, 'rows_v28f': 347, 'rows_removed': 0, 'rows_added': 16}
CAPSCAN regional pairs {'rows_v28d': 56, 'rows_v28f': 60, 'rows_removed': 56, 'rows_added': 60}
Sheet directory {'rows_v28d': 35, 'rows_v28f': 33, 'rows_removed': 8, 'rows_added': 6}
cohort	acquisitions	specimens	participants
CAMPI	15	Defined SIHUMIx mixture; no individua

## Table v30

Every CapScan analysis moves to the 347-acquisition roster (726 acquisitions in all). The five rosters the v28f build left on 331 are re-indexed from the participant-15 reruns: CAPSCAN_FUNCTIONAL_NONDNA15 (participants whose valid protein-and-metabolite specimens cover all five sampling groups; replaces NONDNA14), CAPSCAN_NATIVE13, CAPSCAN_FUNCTIONAL_DNA13 and CAPSCAN_FUNCTIONAL_NONDNA13_SENS (the thirteen participants whose DNA profiles also cover all five groups; replace the twelve-participant rosters), and CAPSCAN_HISTIDINE_ROUTES with its three histidine sheets (fixed-347 A28 run). A valid specimen has an acquisition with positive microbial-OG signal (`CAPSCAN_eligibility_audit_347.tsv`) and a metabolite row (`v30_capscan_metabolite_specimens.tsv`); the recomputed set must equal the specimen roster of the 347 cross-omics run, and the DNA runs must equal the native-profile runs of `source_data/supplementary_methods_8_11/capscan_dna_runs.tsv`. Positive control: the row generator reproduces the published 331 rows of the four cross-omics analyses from their published members. Units: acquisition, specimen, participant, participant-region cell, DNA run. Claim supported: Supplementary Data 1 legend (v30 text paragraph 259), "for all 726 acquisitions".


In [4]:
controls_v30 = panels.reproduce_v28d_crossomics(v28d)
print("positive controls (cross-omics row generator on 331):", controls_v30)
assert all(controls_v30.values())
v30 = panels.build_v30_all(v28f, SD)
panels.write_workbook(OUT / "Sample_rosters_v30.xlsx", order, v30)
panels.write_analysis_index(OUT / "ANALYSIS_INDEX_v30.tsv", v30["Analyses"])
checked_v30 = panels.validate_counts(v30, complete)
entity_v30 = panels.validate_capscan_entity_counts(v30)
changes_v30 = panels.diff_builds(v28f, v30)
print(f"Analyses rows checked {len(checked_v30)}; CapScan entity counts checked {len(entity_v30)}")
for k, v in panels.stat_v30(v30).items():
    print(k, v)


positive controls (cross-omics row generator on 331): {'CAPSCAN_FUNCTIONAL_NONDNA14': True, 'CAPSCAN_NATIVE12': True, 'CAPSCAN_FUNCTIONAL_DNA12': True, 'CAPSCAN_FUNCTIONAL_NONDNA12_SENS': True}


Analyses rows checked 30; CapScan entity counts checked 30
acquisitions_CAMPI 15
acquisitions_Healthy stool 58
acquisitions_PREDICT 306
acquisitions_CapScan 347
acquisitions_total 726
CAPSCAN_SEARCH_ALL 347/308/15/75/0
CAPSCAN_PUBLIC_MG_282 282/250/15/69/0
CAPSCAN_QUANT_NONEMPTY 346/307/15/75/0
CAPSCAN_REGIONAL_DESCRIPTIVE15 346/307/15/75/0
CAPSCAN_REGIONAL_PAIRED15 346/307/15/75/0
CAPSCAN_FUNCTIONAL_NONDNA15 342/303/15/75/0
CAPSCAN_NATIVE13 310/277/13/65/245
CAPSCAN_DNA_COMPLETED301 0/0/15/88/301
CAPSCAN_FUNCTIONAL_DNA13 310/277/13/65/245
CAPSCAN_FUNCTIONAL_NONDNA13_SENS 310/277/13/65/0
CAPSCAN_HISTIDINE_ROUTES 347/308/15/75/Not applicable


## Numbers

Counts behind v28d lines 489, 501 and 503 (units: acquisition, specimen, participant or patient as named). The antibiotic-pair count (42 patients) needs the controlled PREDICT rows.

In [5]:
n = panels.stat_numbers(v28d, v28f, panels.read_tsv(SD / "CAPSCAN_participant15_join.tsv"), disc_v28f, complete)
for k, v in n.items():
    print(k, v if v is not None else "SKIPPED (controlled input absent)")

L489_total_acquisitions_v28d 710
L489_total_acquisitions_v28f 726
L501_campi_acquisitions 15
L501_campi_S11_fractions 4
L501_campi_laboratories 5
L501_hs_acquisitions 58
L501_hs_TechnicalReplicates 12
L501_hs_WorkflowReplicates 15
L501_hs_OnePiece 10
L501_hs_Intra 11
L501_hs_Interindividual 10
L501_hs_excluded_dc 39
L501_predict_acquisitions 306
L501_predict_linked_acquisitions 302
L501_predict_patients 93
L501_predict_mg_comparisons 296
L501_predict_visit1 91
L501_predict_antibiotic_pairs 42
L501_predict_eds85 85
L501_predict_native77 77
L501_capscan_acquisitions 347
L501_capscan_specimens 308
L501_capscan_participants 15
L503_participants_fluid_and_stool 15
L503_p15_fluid_acquisitions 16
L503_discrepant_acquisitions 16
L503_discrepant_in_roster 14


## Checks

The rebuilt v28d workbook is compared with the published one (sheet digests in `expected/SHEET_DIGESTS_v28d.tsv`, computed sheets cell by cell, `ANALYSIS_INDEX.tsv` byte by byte). Receipts go to `checks/` only in the complete build (controlled inputs present); a public build writes them to `build/SData1/`. Then every recomputed number is printed in `NUMBERS.tsv` order.

In [6]:
rdir = UNIT / "checks" if complete else OUT
rdir.mkdir(exist_ok=True)
rec = panels.check_v28d(OUT / "Sample_rosters_v28d.xlsx", OUT / "ANALYSIS_INDEX_v28d.tsv", UNIT / "expected",
                        complete, rdir / "SData1_v28d.json")
print("v28d workbook vs published:", rec["result"], "| ANALYSIS_INDEX byte-identical:", rec["analysis_index_byte_identical"])
for name, v in rec["sheets"].items():
    if v != "IDENTICAL":
        print("  ", name, v, rec["computed_sheet_cell_differences"].get(name, ""))
panels.receipt_v28f(v28f, complete, changes, controls, rdir / "SData1_v28f.json")
panels.receipt_v30(v30, complete, changes_v30, controls_v30, rdir / "SData1_v30.json")
for row in panels.numbers_table(UNIT / "NUMBERS.tsv", n):
    print("\t".join(row))

v28d workbook vs published: IDENTICAL | ANALYSIS_INDEX byte-identical: True
489	710	726	v28f_change:726
501	fifteen	fifteen	MATCH
501	21		OPEN
501	four	four	MATCH
501	five	five	MATCH
501	58	58	MATCH
501	twelve	twelve	MATCH
501	fifteen	fifteen	MATCH
501	ten	ten	MATCH
501	eleven	eleven	MATCH
501	ten	ten	MATCH
501	39	39	MATCH
501	306	306	MATCH
501	302	302	MATCH
501	93	93	MATCH
501	296	296	MATCH
501	91	91	MATCH
501	42	42	MATCH
501	85	85	MATCH
501	77	77	MATCH
501	347	347	MATCH
501	308	308	MATCH
501	fifteen	fifteen	MATCH
503	fifteen	fifteen	MATCH
503	sixteen	sixteen	MATCH
503	Sixteen	Sixteen	MATCH
503	fourteen	fourteen	MATCH
